Here's your plan:

- input → tokens
- token embedding + positional embedding (add them)
- one-head attention ✓ (done)
- multi-head attention — heads in parallel, concat, + an output projection back to n_embd
- feedforward (Linear → ReLU → Linear, 4× wide) ← was missing
- assemble a Block = LN→MHA→+residual, LN→FFN→+residual
- stack N blocks (depth)
- final LayerNorm + lm_head (Linear n_embd → vocab_size to get logits) ← easy to forget
- forward → cross-entropy loss (reshape logits to (B*T, vocab), targets to (B*T,))
- backward + optimizer step (zero_grad → backward → step, AdamW) — this is your "training loop"
- sampling / generate

In [28]:
print("check")

check


In [29]:
import torch 
import torch.nn.functional as F
import torch.nn as nn 

In [30]:
with open('./input.txt','r') as f:
    x = f.read()

In [31]:
T = 4
B = 10
C = 4
heads = 2

In [32]:
temp = torch.randint(1,len(x)-T,(B,))
inp = [x[i.item():i.item()+T] for i in temp]
fout = [x[i.item()+1:i.item()+T+1] for i in temp]
stoi = {}
itos = {}
for i,s in enumerate(sorted(list(set(x)))):
    stoi[s] = i 
    itos[i] = s 


inp2 = []
fout2 = []
for i in inp:
    inp2.append([stoi[j] for j in i])
for i in fout:
    fout2.append([stoi[j] for j in i])

inp2 = torch.tensor(inp2)
inp2

tensor([[42,  1, 58, 46],
        [39,  1, 54, 39],
        [53, 52,  2,  1],
        [43, 47, 56,  1],
        [ 0, 31, 21, 15],
        [57, 43,  6,  1],
        [37, 53, 59,  1],
        [59, 58,  1, 47],
        [59, 56,  1, 57],
        [14, 59, 58,  1]])

In [33]:
fout2

[[1, 58, 46, 43],
 [1, 54, 39, 56],
 [52, 2, 1, 44],
 [47, 56, 1, 39],
 [31, 21, 15, 21],
 [43, 6, 1, 39],
 [53, 59, 1, 57],
 [58, 1, 47, 52],
 [56, 1, 57, 39],
 [59, 58, 1, 44]]

In [34]:
class block(nn.Module):
    def __init__(self):
        super().__init__()
        assert C%heads == 0 , "C must be divisible by heads"
        self.head_size = C//heads
        self.key = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.query = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.value = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.proj = nn.Linear(C,C)
        self.feedfor = nn.Sequential(nn.Linear(C,4*C), nn.ReLU(), nn.Linear(4*C,C))
        self.ln1 = nn.LayerNorm(C)
        self.ln2 = nn.LayerNorm(C)
        



    def attention(self,x):
        x = self.ln1(x)
        out = None
        for i in range(heads):
            query = self.query[i](x)
            key = self.key[i](x)
            value = self.value[i](x)
            
            qk = query@key.transpose(-2,-1) * self.head_size**-0.5
            tril = torch.tril(torch.ones(T,T))
            we = qk.masked_fill(tril == 0, float('-inf'))
            we = torch.softmax(we,dim=-1)
            temp = we@value
            if out is None:
                out = temp
            else:
                out = torch.cat([out,temp],dim=-1)
        return self.proj(out)

    def feedf(self,x):
        x = self.ln2(x)
        return self.feedfor(x)
        
    def forward(self,x):
        att = x+self.attention(x)
        fow = att+self.feedf(att)
        return fow


In [40]:
class GPT(nn.Module):

    def __init__(self,no_of_block):
        super().__init__()
        self.embd = nn.Embedding(len(stoi),C)
        self.posi = nn.Embedding(T,C)
        self.block1 = nn.Sequential(*[block() for _ in range(no_of_block)])
        self.ln3 = nn.LayerNorm(C)
        self.logits = nn.Linear(C,len(stoi))


    def nn_embed(self,x):
        embedings = self.embd(x)
        embedings = embedings+self.posi(torch.arange(T))
        return embedings

    def block(self,e):
        x = self.ln3(self.block1(e))
        x = self.logits(x)
        x = F.cross_entropy(x.view([B*T,len(stoi)]),torch.tensor(fout2).view([-1,]))
        return x



In [47]:
gpt = GPT(3)
optimizer = torch.optim.AdamW(gpt.parameters(),lr=0.001)

In [51]:
for i in range(100):
    emb = gpt.nn_embed(inp2)
    loss = gpt.block(emb)
    print(loss)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()



tensor(4.4165, grad_fn=<NllLossBackward0>)
tensor(4.4057, grad_fn=<NllLossBackward0>)
tensor(4.3955, grad_fn=<NllLossBackward0>)
tensor(4.3860, grad_fn=<NllLossBackward0>)
tensor(4.3770, grad_fn=<NllLossBackward0>)
tensor(4.3681, grad_fn=<NllLossBackward0>)
tensor(4.3591, grad_fn=<NllLossBackward0>)
tensor(4.3501, grad_fn=<NllLossBackward0>)
tensor(4.3411, grad_fn=<NllLossBackward0>)
tensor(4.3320, grad_fn=<NllLossBackward0>)
tensor(4.3229, grad_fn=<NllLossBackward0>)
tensor(4.3139, grad_fn=<NllLossBackward0>)
tensor(4.3050, grad_fn=<NllLossBackward0>)
tensor(4.2960, grad_fn=<NllLossBackward0>)
tensor(4.2871, grad_fn=<NllLossBackward0>)
tensor(4.2783, grad_fn=<NllLossBackward0>)
tensor(4.2695, grad_fn=<NllLossBackward0>)
tensor(4.2609, grad_fn=<NllLossBackward0>)
tensor(4.2522, grad_fn=<NllLossBackward0>)
tensor(4.2435, grad_fn=<NllLossBackward0>)
tensor(4.2347, grad_fn=<NllLossBackward0>)
tensor(4.2257, grad_fn=<NllLossBackward0>)
tensor(4.2168, grad_fn=<NllLossBackward0>)
tensor(4.20

In [46]:
loss

tensor(4.4352, grad_fn=<NllLossBackward0>)

In [13]:
for name, p in gpt.named_parameters():
    print(name, tuple(p.shape))


embd.weight (65, 4)
posi.weight (4, 4)
block1.0.key.0.weight (2, 4)
block1.0.key.1.weight (2, 4)
block1.0.query.0.weight (2, 4)
block1.0.query.1.weight (2, 4)
block1.0.value.0.weight (2, 4)
block1.0.value.1.weight (2, 4)
block1.0.proj.weight (4, 4)
block1.0.proj.bias (4,)
block1.0.feedfor.0.weight (16, 4)
block1.0.feedfor.0.bias (16,)
block1.0.feedfor.2.weight (4, 16)
block1.0.feedfor.2.bias (4,)
block1.0.ln1.weight (4,)
block1.0.ln1.bias (4,)
block1.0.ln2.weight (4,)
block1.0.ln2.bias (4,)
block1.1.key.0.weight (2, 4)
block1.1.key.1.weight (2, 4)
block1.1.query.0.weight (2, 4)
block1.1.query.1.weight (2, 4)
block1.1.value.0.weight (2, 4)
block1.1.value.1.weight (2, 4)
block1.1.proj.weight (4, 4)
block1.1.proj.bias (4,)
block1.1.feedfor.0.weight (16, 4)
block1.1.feedfor.0.bias (16,)
block1.1.feedfor.2.weight (4, 16)
block1.1.feedfor.2.bias (4,)
block1.1.ln1.weight (4,)
block1.1.ln1.bias (4,)
block1.1.ln2.weight (4,)
block1.1.ln2.bias (4,)
block1.2.key.0.weight (2, 4)
block1.2.key.1.wei